# TOAST Load from Config

Author : Henry Nachman

Date : 24 September 2026

---

A tutorial to see if I can load realistic TOAST objects from existing LAT config files. 

In [19]:
import os
import toast
import numpy as np
from toast.observation import default_values as defaults

import sotodlib
import argparse
import h5py

this_dir = os.path.dirname(os.path.abspath("08_toast_from_config.ipynb"))
config_dir = os.path.join(os.path.dirname(this_dir), "mapmaking", "configs")

tele_name = "lat"

In [2]:
parser = argparse.ArgumentParser()
opts = [
    "--config",
    os.path.join(config_dir, "load_context.yml"), # General data loading
    os.path.join(config_dir, "pointing_detector.yml"), # Detector quaternion pointing
    os.path.join(config_dir, "pre_proc.yml"), # Low-level preprocessing
    os.path.join(config_dir, "lat", "pointing.yml"),
    os.path.join(config_dir, "sim_pointing_detector.yml"),
    os.path.join(config_dir, "lat", "sim_pointing.yml"),
    os.path.join(config_dir, "sim_sky_alm.yml"),
    os.path.join(config_dir, "sim_noise.yml"),
    os.path.join(config_dir, "sim_atmosphere.yml"),
    os.path.join(config_dir, "simulation.yml"),
    os.path.join(config_dir, "mapmaking.yml"),
]

In [4]:
det_select = {"wafer.bandpass":"f090","stream_id":"ufm_mv21"}
obs_id = "obs_1761519423_lati1_111"

# context_file = f"/global/cfs/cdirs/sobs/metadata/{tele_name}/contexts/use_this_local.yaml" # NERSC
context_file = f"/cephfs/soukdata/data/tracked/metadata/{tele_name}/contexts/use_this_local.yaml" # SOUK
# context_file = f"/scratch/gpfs/SIMONSOBS/so/tracked/metadata/lat/contexts/use_this_local.yaml"

site_preproc = os.path.join(config_dir, 'lat', "site_proc.yml")


In [5]:
config, otherargs, runargs = toast.config.run_config(parser, opts=opts)
job = toast.traits.create_from_config(config)
job_ops = job.operators

In [6]:
world, procs, rank = toast.mpi.get_world()
comm = toast.mpi.Comm(world=world)
data = toast.Data(comm=comm)

In [7]:
job_ops.load.context_file = context_file
job_ops.load.telescope_name = tele_name
job_ops.load.dets_select = det_select
job_ops.load.observations = [obs_id]
job_ops.load.preprocess_config = site_preproc

In [8]:
job_ops.load.apply(data)

TOAST INFO: load (LoadContext) starting...
TOAST INFO: LoadContext parsed observation sizes in 0.13 s


TOAST INFO: LoadContext obs_1761519423_lati1_111_ufm_mv21 loaded in 6.51 s
TOAST INFO: load (LoadContext) executed in 6.6 s


In [18]:
print(data.obs[0].telescope.focalplane)
print(data.obs[0])

<Focalplane: 695 detectors, sample_rate = 200.00495922941204 Hz, FOV = 4.796201348841562 deg, detectors = [sch_ufm_mv21_1761324909_0_006 .. sch_ufm_mv21_1761325832_6_508]>
<Observation
  name = 'obs_1761519423_lati1_111_ufm_mv21'
  uid = '547870741'  group has 1 processes
  telescope = <Telescope 'lat': uid = 466907968, site = <GroundSite 'SO' : uid = 188141720, lon = -67.78768888888885 deg, lat = -22.960963888888887 deg, alt = 5187.999999999444 m, weather = <toast.weather.Weather object at 0x7fa866320e10>>, focalplane = <Focalplane: 695 detectors, sample_rate = 200.00495922941204 Hz, FOV = 4.796201348841562 deg, detectors = [sch_ufm_mv21_1761324909_0_006 .. sch_ufm_mv21_1761325832_6_508]>>
  session = <Session 'obs_1761519423_lati1_111': uid = 1302040197, start = 2025-10-26 22:57:05.862036+00:00, end = 2025-10-26 23:24:58.746687+00:00>
  obs_info = {'obs_id': 'obs_1761519423_lati1_111', 'timestamp': 1761519425.8620355, 'start_time': 1761519425.8620355, 'stop_time': 1761521098.7466874,

Save the Telescope Object

In [20]:
telescope_save_path = os.path.join(f"{tele_name}_obj.hdf5")
telescope = data.obs[0].telescope
with h5py.File(telescope_save_path, 'w') as f:
    group = f.create_group("telescope")
    telescope.save_hdf5(group)